In [1]:
import sys
sys.path.append("..")

from pathlib import Path

import numpy as np
import pandas as pd
import torch

from src.agents.technical.features import (
    PAIRS,
    TIMEFRAMES,
    add_features,
    load_pair,
    get_feature_names,
 )

try:
    from chronos import ChronosPipeline
except ImportError as exc:
    raise ImportError(
        "chronos-forecasting is required. Install with: pip install chronos-forecasting"
    ) from exc

MODEL_DIR = Path("../models/trained")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CHRONOS_MODEL_IDS = [
    "amazon/chronos-t5-small",
    "amazon/chronos-t5-large",
]

CHRONOS_DEVICE_MAP = "cuda" if torch.cuda.is_available() else "cpu"
CHRONOS_DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# Keep the same feature preparation as Notebook 01 for target/metrics continuity.
data = {}
for pair in PAIRS:
    data[pair] = {}
    for tf in TIMEFRAMES:
        df = load_pair(pair, tf)
        df = add_features(df)
        data[pair][tf] = df

FEATURES = get_feature_names()

print("✓ Data loaded successfully")
print(f"  Pairs     : {PAIRS}")
print(f"  Timeframes: {TIMEFRAMES}")
print(f"  Datasets  : {sum(len(v) for v in data.values())}")
print(f"  Chronos models: {CHRONOS_MODEL_IDS}")
print(f"  Device: {CHRONOS_DEVICE_MAP}, dtype: {CHRONOS_DTYPE}")

✓ Data loaded successfully
  Pairs     : ['EURUSDm', 'GBPUSDm', 'USDJPYm', 'USDCHFm']
  Timeframes: ['D1', 'H4', 'H1']
  Datasets  : 12
  Chronos models: ['amazon/chronos-t5-small', 'amazon/chronos-t5-large']
  Device: cpu, dtype: torch.float32


In [7]:
from dataclasses import dataclass
from typing import Callable

from sklearn.metrics import accuracy_score, mean_squared_error
from skopt.space import Integer, Real

FEATURES = get_feature_names()
RANDOM_STATE = 42

# Set FAST_MODE=True for rapid iteration; set to False for final full benchmark run.
FAST_MODE = True

if FAST_MODE:
    OUTER_MIN_TRAIN = 600
    OUTER_TEST_SIZE = 300
    OUTER_PURGE = 5
    OUTER_EMBARGO = 5
    INNER_SPLITS = 2
    RF_N_ITER_FOLD = 4
    RF_N_ITER_FINAL = 10
    XGB_N_ITER_FOLD = 5
    XGB_N_ITER_FINAL = 10
    RUN_SHAP = False
    SHAP_MAX_SAMPLES = 300
else:
    OUTER_MIN_TRAIN = 600
    OUTER_TEST_SIZE = 120
    OUTER_PURGE = 5
    OUTER_EMBARGO = 5
    INNER_SPLITS = 3
    RF_N_ITER_FOLD = 10
    RF_N_ITER_FINAL = 10
    XGB_N_ITER_FOLD = 10
    XGB_N_ITER_FINAL = 10
    RUN_SHAP = True
    SHAP_MAX_SAMPLES = 1200

ANNUALIZATION = {
    "D1": 252,
    "H4": 252 * 6,
    "H1": 252 * 24,
}


@dataclass
class FoldResult:
    y_true: np.ndarray
    y_pred: np.ndarray
    ret_true: np.ndarray
    ret_pred: np.ndarray


def make_expanding_splits(
    n_samples: int,
    min_train_size: int,
    test_size: int,
    step_size: int | None = None,
    purge_size: int = 1,
    embargo_size: int = 1,
 ) -> list[tuple[np.ndarray, np.ndarray]]:
    """Create chronological expanding splits with explicit purge and embargo."""
    if step_size is None:
        step_size = test_size

    splits = []
    split_train_end = min_train_size

    while True:
        train_end_effective = split_train_end - purge_size
        test_start = split_train_end + embargo_size
        test_end = test_start + test_size

        if train_end_effective <= 0 or test_end > n_samples:
            break

        train_idx = np.arange(0, train_end_effective)
        test_idx = np.arange(test_start, test_end)
        splits.append((train_idx, test_idx))
        split_train_end += step_size

    return splits


def make_inner_walkforward_cv(
    n_train: int,
    n_splits: int = INNER_SPLITS,
    min_train_frac: float = 0.6,
 ) -> list[tuple[np.ndarray, np.ndarray]]:
    """Inner walk-forward CV used by BayesSearchCV (still time-ordered)."""
    if n_train < 80:
        pivot = max(20, int(n_train * 0.7))
        train_idx = np.arange(0, pivot)
        valid_idx = np.arange(pivot, n_train)
        return [(train_idx, valid_idx)]

    start = int(n_train * min_train_frac)
    remaining = n_train - start
    fold_size = max(20, remaining // n_splits)

    cv_splits = []
    train_end = start
    for _ in range(n_splits):
        valid_start = train_end
        valid_end = min(n_train, valid_start + fold_size)
        if valid_end - valid_start < 10:
            break
        tr_idx = np.arange(0, valid_start)
        va_idx = np.arange(valid_start, valid_end)
        cv_splits.append((tr_idx, va_idx))
        train_end = valid_end

    if not cv_splits:
        pivot = max(20, int(n_train * 0.7))
        cv_splits = [(np.arange(0, pivot), np.arange(pivot, n_train))]

    return cv_splits


def safe_mape(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-8) -> float:
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


def sharpe_ratio(returns: np.ndarray, periods_per_year: int) -> float:
    vol = np.std(returns, ddof=1)
    if vol == 0 or np.isnan(vol):
        return 0.0
    return float(np.mean(returns) / vol * np.sqrt(periods_per_year))


def compute_metrics(fold_results: list[FoldResult], periods_per_year: int) -> dict:
    y_true = np.concatenate([f.y_true for f in fold_results])
    y_pred = np.concatenate([f.y_pred for f in fold_results])
    ret_true = np.concatenate([f.ret_true for f in fold_results])
    ret_pred = np.concatenate([f.ret_pred for f in fold_results])

    strategy_returns = (2 * y_pred - 1) * ret_true

    return {
        "rmse": float(np.sqrt(mean_squared_error(ret_true, ret_pred))),
        "mape": safe_mape(ret_true, ret_pred),
        "hit_ratio": float(accuracy_score(y_true, y_pred)),
        "sharpe": sharpe_ratio(strategy_returns, periods_per_year),
        "n_obs": int(len(y_true)),
    }


def evaluate_model_walkforward(
    df: pd.DataFrame,
    tf: str,
    predictor: Callable[[np.ndarray, np.ndarray, np.ndarray], np.ndarray],
    min_train_size: int = OUTER_MIN_TRAIN,
    test_size: int = OUTER_TEST_SIZE,
    purge_size: int = OUTER_PURGE,
    embargo_size: int = OUTER_EMBARGO,
) -> tuple[list[FoldResult], list[tuple[np.ndarray, np.ndarray]]]:
    X = df[FEATURES].to_numpy()
    y = df["target"].to_numpy().astype(int)
    next_ret = df["log_ret"].shift(-1).fillna(0.0).to_numpy()

    n_samples = len(df)
    if n_samples < min_train_size + test_size + purge_size + embargo_size:
        min_train_size = max(200, int(n_samples * 0.5))
        test_size = max(40, int(n_samples * 0.15))

    splits = make_expanding_splits(
        n_samples=n_samples,
        min_train_size=min_train_size,
        test_size=test_size,
        purge_size=purge_size,
        embargo_size=embargo_size,
    )

    fold_results: list[FoldResult] = []
    for train_idx, test_idx in splits:
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        y_pred = predictor(X_train, y_train, X_test).astype(int)

        # Direction-only models are mapped to a return proxy using train fold average magnitude.
        avg_abs_ret = float(np.mean(np.abs(next_ret[train_idx])))
        ret_pred = (2 * y_pred - 1) * avg_abs_ret
        ret_true = next_ret[test_idx]

        fold_results.append(
            FoldResult(
                y_true=y_test,
                y_pred=y_pred,
                ret_true=ret_true,
                ret_pred=ret_pred,
            )
        )

    return fold_results, splits


def save_shap_summary(
    model, X_df: pd.DataFrame, out_path: Path, max_samples: int | None = None
) -> None:
    if max_samples is None:
        max_samples = SHAP_MAX_SAMPLES

    sample = X_df if len(X_df) <= max_samples else X_df.sample(max_samples, random_state=RANDOM_STATE)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(sample)
    if isinstance(shap_values, list):
        shap_values_to_plot = shap_values[1] if len(shap_values) > 1 else shap_values[0]
    else:
        shap_values_to_plot = shap_values

    plt.figure(figsize=(11, 6))
    shap.summary_plot(shap_values_to_plot, sample, show=False, max_display=15)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()


print("✓ Walk-forward utilities ready")
print(f"✓ Feature count: {len(FEATURES)}")
print(f"✓ FAST_MODE: {FAST_MODE}")

✓ Walk-forward utilities ready
✓ Feature count: 15
✓ FAST_MODE: True


## Phase 1A: Chronos Zero-Shot (Small and Large)

Zero-shot setup requirements satisfied here:
- Models: `amazon/chronos-t5-small`, `amazon/chronos-t5-large`
- Input: univariate `close` only
- Forecast: 1-step ahead; median across samples
- Direction rule: `1 if forecast > last_close else 0`

Validation protocol and metrics are reused exactly from Notebook 01 utilities (expanding walk-forward with purge/embargo, RMSE/MAPE/Hit Ratio/Sharpe).

In [ ]:
import time

CHRONOS_CONTEXT_MAX_LEN = 512
CHRONOS_NUM_SAMPLES = 64

# Runtime verbosity controls
VERBOSE_PROGRESS = True
PREVIEW_FIRST_TRY = True
PREVIEW_FIRST_DATASET_ONLY = False

chronos_pipelines: dict[str, ChronosPipeline] = {}

def get_chronos_pipeline(model_id: str) -> ChronosPipeline:
    if model_id not in chronos_pipelines:
        if VERBOSE_PROGRESS:
            print(f"Loading model: {model_id}")
        chronos_pipelines[model_id] = ChronosPipeline.from_pretrained(
            model_id,
            device_map=CHRONOS_DEVICE_MAP,
            dtype=CHRONOS_DTYPE,
        )
    return chronos_pipelines[model_id]


def chronos_median_next_close(
    pipeline: ChronosPipeline,
    history_close: np.ndarray,
    num_samples: int = CHRONOS_NUM_SAMPLES,
    prediction_length: int = 1,
    context_max_len: int = CHRONOS_CONTEXT_MAX_LEN,
) -> float:
    context = np.asarray(history_close[-context_max_len:], dtype=np.float32)
    context_tensor = torch.tensor(context, dtype=torch.float32).unsqueeze(0)
    forecasts = pipeline.predict(
        inputs=context_tensor,
        prediction_length=prediction_length,
        num_samples=num_samples,
    )
    forecast_np = forecasts.detach().cpu().numpy() if torch.is_tensor(forecasts) else np.asarray(forecasts)
    one_step_samples = forecast_np[0, :, 0]
    return float(np.median(one_step_samples))


def make_chronos_zero_shot_predictor(model_id: str):
    pipeline = get_chronos_pipeline(model_id)

    def predictor(train_close: np.ndarray, test_close: np.ndarray) -> np.ndarray:
        history = list(train_close.astype(np.float32))
        yhat_dir = np.zeros(len(test_close), dtype=int)

        for i, observed_close in enumerate(test_close.astype(np.float32)):
            forecast_next = chronos_median_next_close(
                pipeline=pipeline,
                history_close=np.asarray(history, dtype=np.float32),
                num_samples=CHRONOS_NUM_SAMPLES,
                prediction_length=1,
                context_max_len=CHRONOS_CONTEXT_MAX_LEN,
            )
            last_close = float(history[-1])
            yhat_dir[i] = 1 if forecast_next > last_close else 0
            history.append(float(observed_close))

        return yhat_dir

    return predictor


def evaluate_close_model_walkforward(
    df: pd.DataFrame,
    tf: str,
    close_predictor: Callable[[np.ndarray, np.ndarray], np.ndarray],
    min_train_size: int = OUTER_MIN_TRAIN,
    test_size: int = OUTER_TEST_SIZE,
    purge_size: int = OUTER_PURGE,
    embargo_size: int = OUTER_EMBARGO,
) -> tuple[list[FoldResult], list[tuple[np.ndarray, np.ndarray]]]:
    close = df["close"].to_numpy(dtype=np.float32)
    y = df["target"].to_numpy().astype(int)
    next_ret = df["log_ret"].shift(-1).fillna(0.0).to_numpy()

    n_samples = len(df)
    if n_samples < min_train_size + test_size + purge_size + embargo_size:
        min_train_size = max(200, int(n_samples * 0.5))
        test_size = max(40, int(n_samples * 0.15))

    splits = make_expanding_splits(
        n_samples=n_samples,
        min_train_size=min_train_size,
        test_size=test_size,
        purge_size=purge_size,
        embargo_size=embargo_size,
    )

    fold_results: list[FoldResult] = []
    for train_idx, test_idx in splits:
        close_train = close[train_idx]
        close_test = close[test_idx]
        y_test = y[test_idx]

        y_pred = close_predictor(close_train, close_test).astype(int)

        avg_abs_ret = float(np.mean(np.abs(next_ret[train_idx])))
        ret_pred = (2 * y_pred - 1) * avg_abs_ret
        ret_true = next_ret[test_idx]

        fold_results.append(
            FoldResult(
                y_true=y_test,
                y_pred=y_pred,
                ret_true=ret_true,
                ret_pred=ret_pred,
            )
        )

    return fold_results, splits


chronos_zero_shot_rows = []
total_jobs = len(CHRONOS_MODEL_IDS) * len(PAIRS) * len(TIMEFRAMES)
job_no = 0
shown_first_try = False
run_start = time.perf_counter()

for model_id in CHRONOS_MODEL_IDS:
    model_name = model_id.split("/")[-1].replace("-", "_")
    predictor = make_chronos_zero_shot_predictor(model_id)
    if VERBOSE_PROGRESS:
        print(f"\n=== Running zero-shot Chronos: {model_id} ===")

    for pair in PAIRS:
        for tf in TIMEFRAMES:
            job_no += 1
            dataset_start = time.perf_counter()
            if VERBOSE_PROGRESS:
                print(f"[{job_no}/{total_jobs}] {model_name} | {pair} | {tf} ...", end=" ")

            df = data[pair][tf].copy()
            folds, splits = evaluate_close_model_walkforward(
                df=df,
                tf=tf,
                close_predictor=predictor,
                min_train_size=OUTER_MIN_TRAIN,
                test_size=OUTER_TEST_SIZE,
                purge_size=OUTER_PURGE,
                embargo_size=OUTER_EMBARGO,
            )
            metrics = compute_metrics(folds, periods_per_year=ANNUALIZATION[tf])

            if PREVIEW_FIRST_TRY and not shown_first_try and len(folds) > 0:
                first_try_metrics = compute_metrics([folds[0]], periods_per_year=ANNUALIZATION[tf])
                print("\n--- First try preview (first fold only) ---")
                print(
                    f"model={model_name}, pair={pair}, tf={tf}, folds={len(splits)}, "
                    f"fold1_hit={first_try_metrics['hit_ratio']:.4f}, "
                    f"fold1_sharpe={first_try_metrics['sharpe']:.4f}, "
                    f"fold1_rmse={first_try_metrics['rmse']:.6f}, "
                    f"fold1_mape={first_try_metrics['mape']:.4f}"
                )
                shown_first_try = True

            chronos_zero_shot_rows.append(
                {
                    "model": f"chronos_zero_shot_{model_name}",
                    "model_id": model_id,
                    "pair": pair,
                    "timeframe": tf,
                    "folds": len(splits),
                    **metrics,
                }
            )

            elapsed = time.perf_counter() - dataset_start
            if VERBOSE_PROGRESS:
                print(
                    f"done in {elapsed:.1f}s | hit={metrics['hit_ratio']:.4f} "
                    f"sharpe={metrics['sharpe']:.4f} rmse={metrics['rmse']:.6f}"
                )

            if PREVIEW_FIRST_DATASET_ONLY:
                chronos_zero_shot_df = pd.DataFrame(chronos_zero_shot_rows)
                display(chronos_zero_shot_df)
                raise SystemExit("Stopped after first dataset preview by PREVIEW_FIRST_DATASET_ONLY=True")

chronos_zero_shot_df = pd.DataFrame(chronos_zero_shot_rows)
chronos_zero_shot_df = chronos_zero_shot_df.sort_values(["model", "pair", "timeframe"]).reset_index(drop=True)

total_elapsed = time.perf_counter() - run_start
print(f"\n✓ Chronos zero-shot completed in {total_elapsed/60:.2f} min")
display(chronos_zero_shot_df)

Loading model: amazon/chronos-t5-small

=== Running zero-shot Chronos: amazon/chronos-t5-small ===
[1/24] chronos_t5_small | EURUSDm | D1 ... 
--- First try preview (first fold only) ---
model=chronos_t5_small, pair=EURUSDm, tf=D1, folds=3, fold1_hit=0.4900, fold1_sharpe=-1.0188, fold1_rmse=0.005597, fold1_mape=815.4139
done in 1062.4s | hit=0.5222 sharpe=0.4889 rmse=0.005194
[2/24] chronos_t5_small | EURUSDm | H4 ... 